In [ ]:
# --- Configuration ---
# IMPORTANT: Replace this with the actual base path to your images
IMAGE_BASE_PATH = '3A_images/'
HDF5_FILE_NAME = 'alexnet_image_data.h5'
NUM_FOLDS = 5

def load_and_preprocess_image(image_path):
    # This function should load the image and resize it to the expected
    # input size for AlexNet (typically 227x227 or 256x256, usually RGB)
    try:
        # Example using PIL:
        img = Image.open(image_path).convert('RGB')
        # Resize for AlexNet (e.g., 227x227)
        img = img.resize((227, 227))
        # Convert to numpy array and normalize (optional, but good practice)
        img_array = np.array(img, dtype=np.uint8)
        # HDF5 works well with uint8, normalization can be done by the model pipeline
        return img_array

    except FileNotFoundError:
        print(f"Warning: Image not found at {image_path}. Skipping.")
        return None
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return None


# --- Main HDF5 Creation Loop ---
print(f"Starting HDF5 creation: {HDF5_FILE_NAME}")

# Create the HDF5 file
# Note: It's often best practice to use a common image shape, 
# e.g., (227, 227, 3) for AlexNet input.
IMAGE_SHAPE = (227, 227, 3)

with h5py.File(HDF5_FILE_NAME, 'w') as hf:
    
    # Iterate through the 5 folds
    for fold_num in range(NUM_FOLDS):
        
        # Iterate through training (0) and validation (1) data
        for split_index, split_name in enumerate(['train', 'val']):
            
            df = folds[fold_num][split_index]
            
            print(f"\nProcessing Fold {fold_num}, Split: {split_name}...")
            
            # Use a dictionary to hold image data for each class group 
            # to collect them before writing.
            data_by_class = {
                'COVID-19': [],
                'Normal': [],
                'Pneumonia': []
            }
            
            # 1. Iterate through rows in the DataFrame
            for index, row in df.iterrows():
                finding = row['finding']
                image_names = row['images'] # This is a numpy array of image names
                
                # Check if the finding is one of the target classes
                if finding not in data_by_class:
                    print(f"Skipping unknown finding: {finding}")
                    continue
                
                # 2. Iterate through all image names associated with this sample
                if(image_names.size == 1):
                    image_path = os.path.join(IMAGE_BASE_PATH, image_name)
                    
                    # 3. Load and preprocess the image
                    image_data = load_and_preprocess_image(image_path)
                    
                    if image_data is not None and image_data.shape == IMAGE_SHAPE:
                        data_by_class[finding].append(image_data)

                else:
                    for image_name in image_names:
                        image_path = os.path.join(IMAGE_BASE_PATH, image_name)
                    
                        # 3. Load and preprocess the image
                        image_data = load_and_preprocess_image(image_path)
                    
                        if image_data is not None and image_data.shape == IMAGE_SHAPE:
                            data_by_class[finding].append(image_data)
            
            # 4. Write data to HDF5 following the required folder structure
            for class_name, image_list in data_by_class.items():
                if not image_list:
                    print(f"  No images found for {class_name} in this split.")
                    continue
                    
                # Create the HDF5 group path: folds/{fold_number}/{train or val}/{class}
                group_path = f'folds/{fold_num}/{split_name}/{class_name}'
                
                # Stack all images into a single numpy array
                images_array = np.stack(image_list)
                
                # Create and write the dataset
                print(f"  Writing {len(image_list)} images to: {group_path}")
                dset = hf.create_dataset(
                    group_path, 
                    data=images_array, 
                    compression="gzip" # Compression is highly recommended for large image data
                )
                
                # Optional: Store metadata if needed (e.g., original file names)
                # dset.attrs['original_files'] = str(image_names) 

print("\nHDF5 file creation complete!")

# --- Verification Step (Optional) ---
# To check the structure:
# with h5py.File(HDF5_FILE_NAME, 'r') as hf:
#     def print_name(name):
#         print(name)
#     hf.visit(print_name)

In [ ]:
# creating the HDF5 file
with h5py.File('COVID-classification.hdf5','w') as h5f:
    
    # folds
    hdf5_fold = h5f.create_group("folds")

    # groups
    # fold group
    group_1 = h5f.get("folds").create_group("1")
    #group_2 = h5f.get("folds").create_group("2")
    #group_3 = h5f.get("folds").create_group("3")
    #group_4 = h5f.get("folds").create_group("4")
    #group_5 = h5f.get("folds").create_group("5")

    # train and val folders IDÁIG GROUP KELL
    h5f.get("folds/1").create_group("train")
    h5f.get("folds/1").create_group("val")
    
    #h5f.get("folds/2").create_group("train")
    #h5f.get("folds/2").create_group("val")
    
    #h5f.get("folds/3").create_group("train")
    #h5f.get("folds/3").create_group("val")
    
    #h5f.get("folds/4").create_group("train")
    #h5f.get("folds/4").create_group("val")
    
    #h5f.get("folds/5").create_group("train")
    #h5f.get("folds/5").create_group("val")
    
    # train classes itt már DATASET
    h5f.get("folds/1/train").create_dataset("Normal")
    h5f.get("folds/1/train").create_dataset("Pneumonia")
    h5f.get("folds/1/train").create_dataset("COVID-19")
    
    #h5f.get("folds/2/train").create_group("Normal")
    #h5f.get("folds/2/train").create_group("Pneumonia")
    #h5f.get("folds/2/train").create_group("COVID-19")

    #h5f.get("folds/3/train").create_group("Normal")
    #h5f.get("folds/3/train").create_group("Pneumonia")
    #h5f.get("folds/3/train").create_group("COVID-19")

    #h5f.get("folds/4/train").create_group("Normal")
    #h5f.get("folds/4/train").create_group("Pneumonia")
    #h5f.get("folds/4/train").create_group("COVID-19")

    #h5f.get("folds/5/train").create_group("Normal")
    #h5f.get("folds/5/train").create_group("Pneumonia")
    #h5f.get("folds/5/train").create_group("COVID-19")

    # val classes
    h5f.get("folds/1/val").create_group("Normal")
    h5f.get("folds/1/val").create_group("Pneumonia")
    h5f.get("folds/1/val").create_group("COVID-19")

    #h5f.get("folds/2/val").create_group("Normal")
    #h5f.get("folds/2/val").create_group("Pneumonia")
    #h5f.get("folds/2/val").create_group("COVID-19")

    #h5f.get("folds/3/val").create_group("Normal")
    #h5f.get("folds/3/val").create_group("Pneumonia")
    #h5f.get("folds/3/val").create_group("COVID-19")

    #h5f.get("folds/4/val").create_group("Normal")
    #h5f.get("folds/4/val").create_group("Pneumonia")
    #h5f.get("folds/4/val").create_group("COVID-19")

    #h5f.get("folds/5/val").create_group("Normal")
    #h5f.get("folds/5/val").create_group("Pneumonia")
    #h5f.get("folds/5/val").create_group("COVID-19")
 
    h5f.get("folds").create_group("test")
    h5f.get("folds/test").create_group("Normal")
    h5f.get("folds/test").create_group("Pneumonia")
    h5f.get("folds/test").create_group("COVID-19")
    
    h5f.close()

In [ ]:
src = './images/3A_images/'

# loading the TRAINING images into the HDF5 file (images are NOT resized)
# -----------------------------------------------------------------------

#for i in range(1, len(folds) + 1):
for i in range(1,2):
    
    with h5py.File('COVID-classification.hdf5','r+') as h5f:
        
        target = h5f.get("/folds/" + str(i) + "/train")

        print('Loading data into fold: ' + str(i))
        
        # loop trough the TRAINING images
        for item in folds[i-1][0].iterrows():
            
            if(item[1]['finding'] == 'Normal'):
                target = h5f.get("/folds/" + str(i) + "/train/Normal")
            elif(item[1]['finding'] == 'Pneumonia'):
                target = h5f.get("/folds/" + str(i) + "/train/Pneumonia")
            else:
                target = h5f.get("/folds/" + str(i) + "/train/COVID-19")

            if(item[1]['images'].size == 1):
                img = cv2.imread(os.path.join(src, str(item[1]['images'])), cv2.IMREAD_UNCHANGED)
                img_ds = target.create_dataset(str(item[1]['images']), data=img)
            else:
                for j in item[1]['images'].tolist():
                    img = cv2.imread(os.path.join(src,str(j)), cv2.IMREAD_UNCHANGED)
                    img_ds = target.create_dataset(str(j), data=img)

In [ ]:
src = './images/3A_images/'

# loading the VALIDATION images into the HDF5 file (images are NOT resized)
# -------------------------------------------------------------------------

for i in range(1, len(folds) + 1):
    with h5py.File('COVID-classification.hdf5','r+') as h5f:
        
        target = h5f.get("/folds/" + str(i) + "/val")

        print('Loading data into fold: ' + str(i))
        
        # loop trough the VALIDATION images
        for item in folds[i-1][1].iterrows():
            
            if(item[1]['finding'] == 'Normal'):
                target = h5f.get("/folds/" + str(i) + "/val/Normal")
            elif(item[1]['finding'] == 'Pneumonia'):
                target = h5f.get("/folds/" + str(i) + "/val/Pneumonia")
            else:
                target = h5f.get("/folds/" + str(i) + "/val/COVID-19")

            if(item[1]['images'].size == 1):
                img = cv2.imread(os.path.join(src, str(item[1]['images'])), cv2.IMREAD_UNCHANGED)
                img_ds = target.create_dataset(str(item[1]['images']), data=img)
            else:
                for j in item[1]['images'].tolist():
                    img = cv2.imread(os.path.join(src,str(j)), cv2.IMREAD_UNCHANGED)
                    img_ds = target.create_dataset(str(j), data=img)

In [ ]:
import h5py
filename = "COVID-classification.hdf5"

with h5py.File(filename, "r") as f:
    # Print all root level object names (aka keys) 
    # these can be group or dataset names 
    print(f.get('/folds/2/train/Normal'))
    print(f.get('/folds/2/train/Pneumonia'))
    print(f.get('/folds/2/train/COVID-19'))

    print(f.get('/folds/2/val/Normal'))
    print(f.get('/folds/2/val/Pneumonia'))
    print(f.get('/folds/2/val/COVID-19'))
    
    print(f.get('/folds/test/Normal'))
    print(f.get('/folds/test/Pneumonia'))
    print(f.get('/folds/test/COVID-19'))

In [ ]:
src = './images/3A_images/'
# loading the TRAINING images into the HDF5 file (image NOT resized)
with h5py.File('COVID-classification.hdf5','r+') as h5f:
        
    target = h5f.get("/folds/test")
        
    for item in test_full_set.iterrows():
            
        if(item[1]['finding'] == 'Normal'):
            target = h5f.get("/folds/test/Normal")
        elif(item[1]['finding'] == 'Pneumonia'):
            target = h5f.get("/folds/test/Pneumonia")
        else:
            target = h5f.get("/folds/test/COVID-19")

        if(item[1]['images'].size == 1):
            img = cv2.imread(os.path.join(src, str(item[1]['images'])), cv2.IMREAD_UNCHANGED)
            img_ds = target.create_dataset(str(item[1]['images']), data=img)
        else:
            for j in item[1]['images'].tolist():
                img = cv2.imread(os.path.join(src,str(j)), cv2.IMREAD_UNCHANGED)
                img_ds = target.create_dataset(str(j), data=img)

In [ ]:
h5f.close()